# When a naive baseline beats LightGBM: bike-sharing demand with proper time-series CV

The punchline: random 5-fold CV makes LightGBM look like the winner. Time-series 5-fold
says a naive seasonal-mean baseline beats it. Validation strategy matters more than
model choice here.

Place the UCI `hour.csv` at `data/hour.csv` before running.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 110})

## 1. Load

In [ ]:
df = pd.read_csv("data/hour.csv")
df["dteday"] = pd.to_datetime(df["dteday"])
df = df.sort_values(["dteday", "hr"]).reset_index(drop=True)
y = df["cnt"].values

for name, col in [("hour_sin", 2*np.pi*df.hr/24), ("hour_cos", 2*np.pi*df.hr/24),
                  ("month_sin", 2*np.pi*df.mnth/12), ("month_cos", 2*np.pi*df.mnth/12)]:
    df[name] = (np.sin if "sin" in name else np.cos)(col)
df["days_since_start"] = (df["dteday"] - df["dteday"].min()).dt.days

features = ["season","yr","mnth","hr","holiday","weekday","workingday","weathersit",
            "temp","atemp","hum","windspeed","hour_sin","hour_cos","month_sin","month_cos","days_since_start"]
X = df[features].astype(float)
print(X.shape)

## 2. The naive seasonal baseline

Just the mean `cnt` for each (weekday, hour) cell. That's it.

In [ ]:
season_mean = df.groupby(["weekday", "hr"])["cnt"].mean()

def naive_predict(idx):
    keys = list(zip(df.loc[idx, "weekday"], df.loc[idx, "hr"]))
    pred = season_mean.reindex(keys).values
    return np.where(np.isnan(pred), y.mean(), pred)

def rmsle(y_true, y_pred):
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(np.maximum(0, y_pred))))

## 3. Two CV strategies, three models

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
tss = TimeSeriesSplit(n_splits=5)

def cv_model(factory, folds):
    return np.mean([
        rmsle(y[te], factory().fit(X.iloc[tr], y[tr]).predict(X.iloc[te]))
        for tr, te in folds.split(X)
    ])

def cv_naive(folds):
    return np.mean([rmsle(y[te], naive_predict(te)) for _, te in folds.split(X)])

ridge_f = lambda: Ridge(alpha=1.0)
lgbm_f = lambda: LGBMRegressor(n_estimators=600, num_leaves=31, learning_rate=0.05, verbose=-1, random_state=42)

results = pd.DataFrame({
    "Random 5-fold": [cv_naive(kf), cv_model(ridge_f, kf), cv_model(lgbm_f, kf)],
    "Time-series 5-fold": [cv_naive(tss), cv_model(ridge_f, tss), cv_model(lgbm_f, tss)],
}, index=["Naive seasonal", "Ridge", "LightGBM"]).round(4)
results

## 4. What this does and doesn't mean

LightGBM "losing" to the naive baseline under time-series CV doesn't mean it's a bad
model. It means the seasonal signal is so strong that engineered weather + cyclical
features add little over the (weekday, hour) mean when you evaluate honestly. In
practice, the right move is to model residuals against the naive baseline rather
than the target directly.

Full write-up: <https://ndjstn.github.io/posts/bike-sharing-timeseries-cv/>.